In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from itertools import product
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             classification_report)
from collections import Counter
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder


# Adjust to your actual structure inside ML_Final_Project
BASE_DIR = '/content/drive/MyDrive/ML_Final_Project'
DATA_DIR = os.path.join(BASE_DIR, 'data')   # or wherever your data lives
MODEL_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = os.path.join(MODEL_DIR, 'asl_cnn.pth')

print("Data dir contents:", os.listdir(DATA_DIR))

Data dir contents: ['X_train.npy', 'X_val.npy', 'X_test.npy', 'y_train.npy', 'classes.npy', 'y_test.npy', 'y_val.npy', 'train_losses.npy', 'train_accs.npy', 'val_losses.npy', 'val_accs.npy', 'y_pred.npy', 'y_probs.npy', 'external_y_pred.npy', 'external_y_true.npy']


In [ ]:
# Cell 2: Majority Class Baseline
# Assumes Cell 0 has set BASE_DIR and DATA_DIR



# DATA_DIR comes from Cell 0:
# DATA_DIR = os.path.join(BASE_DIR, 'processed_data')

print("=" * 50)
print("Loading data...")
print("=" * 50)

y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))
classes = np.load(os.path.join(DATA_DIR, "classes.npy"), allow_pickle=True)

print(f"Training samples: {len(y_train)}")
print(f"Test samples:     {len(y_test)}")
print(f"Classes:          {classes}")

print("\n" + "=" * 50)
print("Running Majority Class Predictor...")
print("=" * 50)

most_common_class = Counter(y_train).most_common(1)[0][0]
most_common_label = classes[most_common_class]
print(f"Most common class in training set: '{most_common_label}' (index {most_common_class})")

# Predict the majority class for every test sample
y_pred_majority = np.full(len(y_test), most_common_class)

print("\n" + "=" * 50)
print("MAJORITY CLASS BASELINE RESULTS")
print("=" * 50)

acc  = accuracy_score(y_test, y_pred_majority)
prec = precision_score(y_test, y_pred_majority, average="macro", zero_division=0)
rec  = recall_score(y_test, y_pred_majority, average="macro", zero_division=0)
f1   = f1_score(y_test, y_pred_majority, average="macro", zero_division=0)

print(f"Accuracy:  {acc:.4f}  ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f} (macro)")
print(f"Recall:    {rec:.4f}  (macro)")
print(f"F1 Score:  {f1:.4f}  (macro)")

print("\nNote: With 29 balanced classes, random chance = ~3.45%")
print("This baseline confirms our dataset is balanced and")
print("highlights how much better our CNN performs.")

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_majority,
    target_names=classes.astype(str),
    zero_division=0
))

Loading data...
Training samples: 56700
Test samples:     12150
Classes:          ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S'
 'T' 'U' 'V' 'W' 'X' 'Y' 'del' 'nothing' 'space']

Running Majority Class Predictor...
Most common class in training set: 'M' (index 11)

MAJORITY CLASS BASELINE RESULTS
Accuracy:  0.0370  (3.70%)
Precision: 0.0014 (macro)
Recall:    0.0370  (macro)
F1 Score:  0.0026  (macro)

Note: With 29 balanced classes, random chance = ~3.45%
This baseline confirms our dataset is balanced and
highlights how much better our CNN performs.

Classification Report:
              precision    recall  f1-score   support

           A       0.00      0.00      0.00       450
           B       0.00      0.00      0.00       450
           C       0.00      0.00      0.00       450
           D       0.00      0.00      0.00       450
           E       0.00      0.00      0.00       450
           F       0.00      0.00      0.00       450
           G 

# CNN baseline

In [ ]:
#Cell 3: CNN Baseline (1 block)
# Assumes Cell 0 has set BASE_DIR, DATA_DIR, MODEL_DIR, DEVICE



# Hyperparameters
EPOCHS        = 10
BATCH_SIZE    = 32
LEARNING_RATE = 0.001



print("\n" + "=" * 50)
print("STEP 1: Loading preprocessed data...")
print("=" * 50)

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))
classes = np.load(os.path.join(DATA_DIR, "classes.npy"), allow_pickle=True)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

print("\n" + "=" * 50)
print("STEP 2: Preparing tensors...")
print("=" * 50)

X_train = np.transpose(X_train, (0, 3, 1, 2))
X_test  = np.transpose(X_test,  (0, 3, 1, 2))

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                          batch_size=BATCH_SIZE, shuffle=True)

print("\n" + "=" * 50)
print("STEP 3: Building Simple 1-Block CNN baseline...")
print("=" * 50)

num_classes = len(classes)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.fc(x)
        return x


model     = SimpleCNN(num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

print("\n" + "=" * 50)
print("STEP 4: Training Simple CNN baseline...")
print("=" * 50)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total       += labels.size(0)
        correct     += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total
    print(f"Epoch [{epoch+1:2d}/{EPOCHS}] "
          f"Loss: {running_loss/len(train_loader):.4f} | "
          f"Train Acc: {train_acc:.2f}%")

print("\n" + "=" * 50)
print("STEP 5: Evaluating Simple CNN baseline...")
print("=" * 50)

model.eval()
all_preds = []

with torch.no_grad():
    for i in range(0, len(X_test_t), BATCH_SIZE):
        batch    = X_test_t[i:i+BATCH_SIZE].to(DEVICE)
        outputs  = model(batch)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())

all_preds = np.array(all_preds)

acc  = accuracy_score(y_test, all_preds)
prec = precision_score(y_test, all_preds, average="macro", zero_division=0)
rec  = recall_score(y_test, all_preds, average="macro", zero_division=0)
f1   = f1_score(y_test, all_preds, average="macro", zero_division=0)

print("\n" + "=" * 50)
print("SIMPLE CNN BASELINE RESULTS")
print("=" * 50)
print(f"Accuracy:  {acc:.4f}  ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f} (macro)")
print(f"Recall:    {rec:.4f}  (macro)")
print(f"F1 Score:  {f1:.4f}  (macro)")

print("\nClassification Report:")
print(classification_report(
    y_test, all_preds,
    target_names=classes.astype(str),
    zero_division=0
))

# Save model to Drive (persists across Colab disconnects)
torch.save(model.state_dict(), os.path.join(MODEL_DIR, "simple_cnn_baseline.pth"))
print(f"\nBaseline model saved to {MODEL_DIR}/simple_cnn_baseline.pth")


STEP 1: Loading preprocessed data...
X_train: (56700, 64, 64, 3)
X_test:  (12150, 64, 64, 3)

STEP 2: Preparing tensors...

STEP 3: Building Simple 1-Block CNN baseline...
SimpleCNN(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=512, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=27, bias=True)
  )
)

STEP 4: Training Simple CNN baseline...
Epoch [ 1/10] Loss: 2.8743 | Train Acc: 11.93%
Epoch [ 2/10] Loss: 2.5109 | Train Acc: 16.61%
Epoch [ 3/10] Loss: 2.3470 | Train Acc: 20.45%
Epoch [ 4/10] Loss: 2.2409 | Train Acc: 23.52%
Epoch [ 5/10] Loss: 2.1516 | Train Acc: 25.86%
Ep

# Hyperparameter Tuning


In [ ]:
# Cell 4: Hyperparameter Tuning (Grid Search)
# Assumes Cell 0 has set BASE_DIR, DATA_DIR, MODEL_DIR, DEVICE


print(f"Using device: {DEVICE}")

# ── HYPERPARAMETER GRID ────────────────────────────────────────────────
PARAM_GRID = {
    "learning_rate": [ 0.001, 0.0001],
    "batch_size":    [32, 64],
    "epochs":        [5, 10]
}

# ── LOAD DATA ──────────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("Loading data...")
print("=" * 50)

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
X_val   = np.load(os.path.join(DATA_DIR, "X_val.npy"))
y_val   = np.load(os.path.join(DATA_DIR, "y_val.npy"))
classes = np.load(os.path.join(DATA_DIR, "classes.npy"), allow_pickle=True)

X_train = np.transpose(X_train, (0, 3, 1, 2))
X_val   = np.transpose(X_val,   (0, 3, 1, 2))

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)

num_classes = len(classes)
print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")


# ── MODEL ──────────────────────────────────────────────────────────────
class ASL_CNN(nn.Module):
    def __init__(self, num_classes):
        super(ASL_CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x


# ── TRAIN + EVALUATE FUNCTION ──────────────────────────────────────────
def train_and_evaluate(lr, batch_size, epochs):
    model     = ASL_CNN(num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size,
        shuffle=True
    )

    # Training
    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

    # Validation
    model.eval()
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(X_val_t), batch_size):
            batch    = X_val_t[i:i+batch_size].to(DEVICE)
            _, preds = model(batch).max(1)
            all_preds.extend(preds.cpu().numpy())

    acc = accuracy_score(y_val, np.array(all_preds))
    return acc, model


# ── GRID SEARCH ────────────────────────────────────────────────────────
total_combos = (len(PARAM_GRID['learning_rate']) *
                len(PARAM_GRID['batch_size']) *
                len(PARAM_GRID['epochs']))

print("\n" + "=" * 50)
print("Starting Grid Search...")
print(f"Total combinations: {total_combos}")
print("=" * 50)

results     = []
best_acc    = 0.0
best_params = {}
best_model  = None
combo_num   = 0

# Path for saving progress as we go (so a Colab disconnect doesn't kill everything)
RESULTS_PATH = os.path.join(MODEL_DIR, "tuning_results.json")

for lr, bs, ep in product(PARAM_GRID["learning_rate"],
                          PARAM_GRID["batch_size"],
                          PARAM_GRID["epochs"]):
    combo_num += 1
    print(f"\nCombo {combo_num}/{total_combos}: lr={lr} | batch_size={bs} | epochs={ep}")

    acc, model = train_and_evaluate(lr, bs, ep)
    results.append({"lr": lr, "batch_size": bs, "epochs": ep, "val_acc": float(acc)})

    print(f"  → Val Accuracy: {acc*100:.2f}%")

    if acc > best_acc:
        best_acc    = acc
        best_params = {"learning_rate": lr, "batch_size": bs, "epochs": ep}
        best_model  = model
        # Save best model to Drive every time it improves (in case of disconnect)
        torch.save(best_model.state_dict(),
                   os.path.join(MODEL_DIR, "asl_cnn_best_tuned.pth"))
        print(f"  → New best! Saved checkpoint.")

    # Save running results to Drive after each combo
    with open(RESULTS_PATH, "w") as f:
        json.dump({"results": results, "best_params": best_params,
                   "best_acc": float(best_acc)}, f, indent=2)


# ── FINAL REPORT ───────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("GRID SEARCH RESULTS")
print("=" * 50)
print(f"{'LR':<10} {'Batch':<10} {'Epochs':<10} {'Val Acc':<10}")
print("-" * 40)
for r in sorted(results, key=lambda x: x["val_acc"], reverse=True):
    print(f"{r['lr']:<10} {r['batch_size']:<10} {r['epochs']:<10} {r['val_acc']*100:.2f}%")

print("\n" + "=" * 50)
print("BEST HYPERPARAMETERS FOUND")
print("=" * 50)
print(f"Learning Rate: {best_params['learning_rate']}")
print(f"Batch Size:    {best_params['batch_size']}")
print(f"Epochs:        {best_params['epochs']}")
print(f"Val Accuracy:  {best_acc*100:.2f}%")

print(f"\nBest model saved to: {MODEL_DIR}/asl_cnn_best_tuned.pth")
print(f"Full results saved to: {RESULTS_PATH}")

Using device: cuda

Loading data...
X_train: (56700, 3, 64, 64) | X_val: (12150, 3, 64, 64)

Starting Grid Search...
Total combinations: 8

Combo 1/8: lr=0.001 | batch_size=32 | epochs=5
  → Val Accuracy: 98.86%
  → New best! Saved checkpoint.

Combo 2/8: lr=0.001 | batch_size=32 | epochs=10
  → Val Accuracy: 99.65%
  → New best! Saved checkpoint.

Combo 3/8: lr=0.001 | batch_size=64 | epochs=5
  → Val Accuracy: 98.13%

Combo 4/8: lr=0.001 | batch_size=64 | epochs=10
  → Val Accuracy: 99.40%

Combo 5/8: lr=0.0001 | batch_size=32 | epochs=5
  → Val Accuracy: 99.36%

Combo 6/8: lr=0.0001 | batch_size=32 | epochs=10
  → Val Accuracy: 99.78%
  → New best! Saved checkpoint.

Combo 7/8: lr=0.0001 | batch_size=64 | epochs=5
  → Val Accuracy: 98.92%

Combo 8/8: lr=0.0001 | batch_size=64 | epochs=10
  → Val Accuracy: 99.87%
  → New best! Saved checkpoint.

GRID SEARCH RESULTS
LR         Batch      Epochs     Val Acc   
----------------------------------------
0.0001     64         10         99

# Training

In [ ]:
# Cell 5: Train the main CNN model
# Assumes Cell 0 has set BASE_DIR, DATA_DIR, MODEL_DIR, DEVICE


# Hyperparameters — adjust based on hyperparameter_tuning results
IMG_SIZE      = 64
BATCH_SIZE    = 64
EPOCHS        = 10
LEARNING_RATE = 0.0001

print(f"Using device: {DEVICE}")

# ── STEP 1: LOAD PREPROCESSED DATA ───────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 1: Loading preprocessed data...")
print("=" * 50)

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
X_val   = np.load(os.path.join(DATA_DIR, "X_val.npy"))
y_val   = np.load(os.path.join(DATA_DIR, "y_val.npy"))

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")

# ── STEP 2: PREPARE TENSORS ───────────────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 2: Preparing PyTorch tensors...")
print("=" * 50)

# PyTorch expects (batch, channels, height, width)
X_train = np.transpose(X_train, (0, 3, 1, 2))
X_val   = np.transpose(X_val,   (0, 3, 1, 2))

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset   = TensorDataset(X_val_t,   y_val_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# ── STEP 3: BUILD CNN MODEL ───────────────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 3: Building CNN model...")
print("=" * 50)

num_classes = len(np.load(os.path.join(DATA_DIR, "classes.npy"), allow_pickle=True))
print(f"Number of classes: {num_classes}")

class ASL_CNN(nn.Module):
    def __init__(self, num_classes):
        super(ASL_CNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)                            # 64 → 32
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)                            # 32 → 16
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)                            # 16 → 8
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x


model = ASL_CNN(num_classes).to(DEVICE)
print(model)

# ── STEP 4: LOSS + OPTIMIZER ──────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ── STEP 5: TRAINING LOOP ─────────────────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 5: Training...")
print("=" * 50)

MODEL_PATH       = os.path.join(MODEL_DIR, "asl_cnn.pth")
BEST_MODEL_PATH  = os.path.join(MODEL_DIR, "asl_cnn_best.pth")

train_losses = []
val_losses   = []
train_accs   = []
val_accs     = []

best_val_acc = 0.0

for epoch in range(EPOCHS):

    # ── TRAIN ──────────────────────────────────────────
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total       += labels.size(0)
        correct     += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc  = 100. * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # ── VALIDATE ───────────────────────────────────────
    model.eval()
    val_loss = 0.0
    correct  = 0
    total    = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs        = model(images)
            loss           = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total       += labels.size(0)
            correct     += predicted.eq(labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc  = 100. * correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch+1:2d}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Save checkpoint to Drive every epoch (survives Colab disconnects)
    torch.save(model.state_dict(), MODEL_PATH)

    # Track best model separately
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"           → New best val acc! Saved to {BEST_MODEL_PATH}")

    # Save running metrics after every epoch too
    np.save(os.path.join(DATA_DIR, "train_losses.npy"), np.array(train_losses))
    np.save(os.path.join(DATA_DIR, "val_losses.npy"),   np.array(val_losses))
    np.save(os.path.join(DATA_DIR, "train_accs.npy"),   np.array(train_accs))
    np.save(os.path.join(DATA_DIR, "val_accs.npy"),     np.array(val_accs))

# ── STEP 6: FINAL REPORT ──────────────────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 6: Training complete!")
print("=" * 50)

print(f"Final Train Accuracy: {train_accs[-1]:.2f}%")
print(f"Final Val Accuracy:   {val_accs[-1]:.2f}%")
print(f"Best Val Accuracy:    {best_val_acc:.2f}%")
print(f"\nFinal model saved to:  {MODEL_PATH}")
print(f"Best model saved to:   {BEST_MODEL_PATH}")

Using device: cuda

STEP 1: Loading preprocessed data...
X_train: (56700, 64, 64, 3)
X_val:   (12150, 64, 64, 3)

STEP 2: Preparing PyTorch tensors...
Training batches:   886
Validation batches: 190

STEP 3: Building CNN model...
Number of classes: 27
ASL_CNN(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=T

# Testing


In [ ]:
# Cell 6: Evaluate the trained CNN on the test set
# Assumes Cell 0 has set BASE_DIR, DATA_DIR, MODEL_DIR, DEVICE


# Path to the model — use the BEST checkpoint from training
MODEL_PATH = os.path.join(MODEL_DIR, "asl_cnn_best.pth")
# Or use asl_cnn.pth if you want the final-epoch version:
# MODEL_PATH = os.path.join(MODEL_DIR, "asl_cnn.pth")

BATCH_SIZE = 64

print(f"Using device: {DEVICE}")
print(f"Loading model from: {MODEL_PATH}")

# ── STEP 1: LOAD TEST DATA ─────────────────────────────────────────────────
print("\nLoading test data...")

X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))
classes = np.load(os.path.join(DATA_DIR, "classes.npy"), allow_pickle=True)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Classes: {classes}")

# (batch, channels, height, width)
X_test = np.transpose(X_test, (0, 3, 1, 2))

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# ── STEP 2: RECREATE THE MODEL ARCHITECTURE ────────────────────────────────
class ASL_CNN(nn.Module):
    def __init__(self, num_classes):
        super(ASL_CNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x

num_classes = len(classes)
model = ASL_CNN(num_classes).to(DEVICE)

# ── STEP 3: LOAD SAVED WEIGHTS ─────────────────────────────────────────────
print("\nLoading model weights...")
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

# ── STEP 4: RUN INFERENCE (batched) ────────────────────────────────────────
print("\nRunning predictions on test set...")

all_preds = []
all_probs = []

with torch.no_grad():
    for i in range(0, len(X_test_t), BATCH_SIZE):
        batch = X_test_t[i:i+BATCH_SIZE].to(DEVICE)
        outputs = model(batch)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_preds.append(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

y_pred = np.concatenate(all_preds)
y_probs = np.concatenate(all_probs)

# ── STEP 5: COMPUTE METRICS ────────────────────────────────────────────────
acc        = accuracy_score(y_test, y_pred)
prec_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec_macro  = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro   = f1_score(y_test, y_pred, average="macro", zero_division=0)

print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec_macro:.4f} (macro)")
print(f"Recall:    {rec_macro:.4f} (macro)")
print(f"F1 score:  {f1_macro:.4f} (macro)")

# ── STEP 6: CONFUSION MATRIX + REPORT ──────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=[str(c) for c in classes],
    zero_division=0
))

# ── STEP 7: SHOW A FEW MISTAKES ────────────────────────────────────────────
wrong = np.where(y_pred != y_test)[0]
print(f"\nMisclassified samples: {len(wrong)} / {len(y_test)}")
if len(wrong) > 0:
    print("First 10 mistakes:")
    for idx in wrong[:10]:
        true_label = classes[y_test[idx]]
        pred_label = classes[y_pred[idx]]
        confidence = y_probs[idx][y_pred[idx]] * 100
        print(f"  sample {idx}: true={true_label}, pred={pred_label} ({confidence:.1f}% confidence)")

# ── STEP 8: SAVE PREDICTIONS TO DRIVE (optional) ───────────────────────────
np.save(os.path.join(DATA_DIR, "y_pred.npy"),  y_pred)
np.save(os.path.join(DATA_DIR, "y_probs.npy"), y_probs)
print(f"\nPredictions saved to {DATA_DIR}/y_pred.npy and y_probs.npy")

Using device: cuda
Loading model from: /content/drive/MyDrive/ML_Final_Project/models/asl_cnn_best.pth

Loading test data...
X_test shape: (12150, 64, 64, 3)
y_test shape: (12150,)
Classes: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S'
 'T' 'U' 'V' 'W' 'X' 'Y' 'del' 'nothing' 'space']

Loading model weights...

Running predictions on test set...

TEST SET RESULTS
Accuracy:  0.9985
Precision: 0.9985 (macro)
Recall:    0.9985 (macro)
F1 score:  0.9985 (macro)

Confusion Matrix:
[[448   1   0   0   1   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0]
 [  0 450   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0]
 [  0   0 450   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0]
 [  0   0   0 450   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0]
 [  0   0   0   0 450   0   0   0   0   

# Testing new dataset

In [ ]:
# Cell 7: External Generalization Evaluation
# Tests the trained model on a DIFFERENT ASL dataset than it was trained on
# Assumes Cell 0 has set BASE_DIR, DATA_DIR, MODEL_DIR, DEVICE


# ── CONFIG ─────────────────────────────────────────────────────────────────
IMG_SIZE       = 64
BATCH_SIZE     = 128
MAX_PER_CLASS  = 500   # cap per class so big external datasets don't crash RAM

# CHOOSE ONE: where does the external dataset come from?
USE_KAGGLEHUB  = True
KAGGLE_DATASET = "debashishsau/aslamerican-sign-language-aplhabet-dataset"

# If USE_KAGGLEHUB is False, set this to your external dataset folder in Drive
# e.g. os.path.join(BASE_DIR, "external_dataset")
EXTERNAL_DIR_OVERRIDE = None

# Path to the trained model — pick the best checkpoint
MODEL_PATH = os.path.join(MODEL_DIR, "asl_cnn_best.pth")
# Or final-epoch version:
# MODEL_PATH = os.path.join(MODEL_DIR, "asl_cnn.pth")

print(f"Using device: {DEVICE}")
print(f"Model path:   {MODEL_PATH}")

# ── HELPERS ────────────────────────────────────────────────────────────────
def load_images(folder_path, allowed_classes=None, max_per_class=200):
    images = []
    labels = []
    class_counts = {}

    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Dataset folder not found: {folder_path}")

    for label in os.listdir(folder_path):
        label_path = os.path.join(folder_path, label)

        if not os.path.isdir(label_path):
            continue

        if allowed_classes is not None and label not in allowed_classes:
            continue

        class_counts[label] = 0

        for img_file in os.listdir(label_path):
            if class_counts[label] >= max_per_class:
                break

            img_path = os.path.join(label_path, img_file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0
            images.append(img)
            labels.append(label)
            class_counts[label] += 1

    return np.array(images), np.array(labels)


def resolve_image_root(path):
    """Find the subfolder that contains the actual class directories."""
    for root, dirs, files in os.walk(path):
        subdirs = [d for d in dirs if os.path.isdir(os.path.join(root, d))]
        if len(subdirs) >= 10:
            return root
    return path


# ── MODEL ARCHITECTURE ─────────────────────────────────────────────────────
class ASL_CNN(nn.Module):
    def __init__(self, num_classes):
        super(ASL_CNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x


# ── LOAD CLASS LIST (from original training preprocessing) ─────────────────
classes_path = os.path.join(DATA_DIR, "classes.npy")
if not os.path.exists(classes_path):
    raise FileNotFoundError(f"Could not find classes at: {classes_path}")

classes = np.load(classes_path, allow_pickle=True)
known_classes = set(classes.tolist())
num_classes = len(classes)
print(f"Known classes ({num_classes}): {classes}")

# ── GET EXTERNAL DATASET ───────────────────────────────────────────────────
if USE_KAGGLEHUB:
    # Install kagglehub if not already (Colab usually has it)
    import subprocess, sys
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub

    print(f"\nDownloading external dataset: {KAGGLE_DATASET}")
    raw_path = kagglehub.dataset_download(KAGGLE_DATASET)
    external_dir = resolve_image_root(raw_path)
    print(f"Downloaded to:     {raw_path}")
    print(f"Resolved root:     {external_dir}")
else:
    if EXTERNAL_DIR_OVERRIDE is None:
        raise ValueError("Set EXTERNAL_DIR_OVERRIDE or set USE_KAGGLEHUB=True")
    external_dir = EXTERNAL_DIR_OVERRIDE
    print(f"\nUsing external dataset from: {external_dir}")

# ── LOAD EXTERNAL IMAGES ───────────────────────────────────────────────────
print("\nLoading external images...")
X_ext_raw, y_ext_raw = load_images(
    external_dir,
    allowed_classes=known_classes,
    max_per_class=MAX_PER_CLASS
)

if len(X_ext_raw) == 0:
    raise ValueError(
        f"No images found in {external_dir}. "
        "Check the folder path and make sure it contains class subfolders."
    )

print(f"External images: {X_ext_raw.shape}")
print(f"External labels: {y_ext_raw.shape}")
print(f"Classes kept:    {np.unique(y_ext_raw)}")

# Encode labels using the SAME order as training
encoder = LabelEncoder()
encoder.fit(classes)
y_ext = encoder.transform(y_ext_raw)

# (batch, channels, height, width)
X_ext = np.transpose(X_ext_raw, (0, 3, 1, 2))
X_ext_t = torch.tensor(X_ext, dtype=torch.float32)
y_ext_t = torch.tensor(y_ext, dtype=torch.long)

test_loader = DataLoader(
    TensorDataset(X_ext_t, y_ext_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ── LOAD MODEL ─────────────────────────────────────────────────────────────
model = ASL_CNN(num_classes).to(DEVICE)
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

# ── INFERENCE ──────────────────────────────────────────────────────────────
all_preds = []
all_true  = []

print("\nRunning external evaluation...")
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_true.extend(labels.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

# ── METRICS ────────────────────────────────────────────────────────────────
acc        = accuracy_score(all_true, all_preds)
prec_macro = precision_score(all_true, all_preds, average="macro", zero_division=0)
rec_macro  = recall_score(all_true, all_preds, average="macro", zero_division=0)
f1_macro   = f1_score(all_true, all_preds, average="macro", zero_division=0)

print("\n" + "=" * 60)
print("EXTERNAL GENERALIZATION RESULTS")
print("=" * 60)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec_macro:.4f} (macro)")
print(f"Recall:    {rec_macro:.4f} (macro)")
print(f"F1 score:  {f1_macro:.4f} (macro)")

cm = confusion_matrix(all_true, all_preds)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    all_true, all_preds,
    target_names=classes.astype(str),
    zero_division=0
))

wrong = np.where(all_preds != all_true)[0]
print(f"\nMisclassified samples: {len(wrong)} / {len(all_true)}")
if len(wrong) > 0:
    print("First 10 mistakes:")
    for idx in wrong[:10]:
        true_label = classes[all_true[idx]]
        pred_label = classes[all_preds[idx]]
        print(f"  sample {idx}: true={true_label}, pred={pred_label}")

# Save external eval results to Drive
np.save(os.path.join(DATA_DIR, "external_y_pred.npy"), all_preds)
np.save(os.path.join(DATA_DIR, "external_y_true.npy"), all_true)
print(f"\nExternal predictions saved to {DATA_DIR}/external_y_pred.npy")

Using device: cuda
Model path:   /content/drive/MyDrive/ML_Final_Project/models/asl_cnn_best.pth
Known classes (27): ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S'
 'T' 'U' 'V' 'W' 'X' 'Y' 'del' 'nothing' 'space']

Using Colab cache for faster access to the 'aslamerican-sign-language-aplhabet-dataset' dataset.
Downloaded to:     /kaggle/input/aslamerican-sign-language-aplhabet-dataset
Resolved root:     /kaggle/input/aslamerican-sign-language-aplhabet-dataset/ASL_Alphabet_Dataset/asl_alphabet_train

Loading external images...
External images: (13500, 64, 64, 3)
External labels: (13500,)
Classes kept:    ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S'
 'T' 'U' 'V' 'W' 'X' 'Y' 'del' 'nothing' 'space']

Running external evaluation...

EXTERNAL GENERALIZATION RESULTS
Accuracy:  0.7789
Precision: 0.8474 (macro)
Recall:    0.7789 (macro)
F1 score:  0.7954 (macro)

Confusion Matrix:
[[342   1   0   0   1  28   1   0   0   0  13  47  24   4  1

# Webcam tester

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import pyautogui
import time

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_DIR     = "models"
DATA_DIR      = "processed_data"
IMG_SIZE      = 64
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIDENCE    = 0.85      # minimum confidence to accept a prediction
COOLDOWN      = 3       # seconds between typing each letter
BOX_SIZE      = 350       # size of the hand capture box

# ── LOAD MODEL ────────────────────────────────────────────────────────────────
class ASL_CNN(nn.Module):
    def _init_(self, num_classes):
        super(ASL_CNN, self)._init_()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x


classes     = np.load(f"{DATA_DIR}/classes.npy")
num_classes = len(classes)

model = ASL_CNN(num_classes).to(DEVICE)
model.load_state_dict(torch.load(f"{MODEL_DIR}/asl_cnn.pth",
                                  map_location=DEVICE))
model.eval()
print("Model loaded!")
print(f"Classes: {classes}")

# ── HELPER: PREPROCESS FRAME ──────────────────────────────────────────────────
def preprocess(roi):
    img = cv2.resize(roi, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.transpose(img, (2, 0, 1))           # HWC → CHW
    img = torch.tensor(img, dtype=torch.float32)
    img = img.unsqueeze(0).to(DEVICE)            # add batch dimension
    return img

# ── HELPER: PREDICT ───────────────────────────────────────────────────────────
def predict(roi):
    tensor = preprocess(roi)
    with torch.no_grad():
        outputs     = model(tensor)
        probs       = torch.softmax(outputs, dim=1)
        confidence, predicted = probs.max(1)
    return classes[predicted.item()], confidence.item()

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
cap           = cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
if not cap.isOpened():
    print("ERROR: Could not open camera")
    exit()
last_typed    = time.time()
current_word  = ""

print("\n🤟 Sign Language Translator Running!")
print("Place your hand inside the GREEN box")
print("Press 'Q' to quit | Press 'SPACE' to add space | Press 'BACKSPACE' to delete\n")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame, retrying...")
        time.sleep(0.1)
        continue

    frame = cv2.flip(frame, 1)                   # mirror the webcam
    h, w  = frame.shape[:2]

    # Define the hand capture box in the center-right of the frame
    x1 = w // 2 - BOX_SIZE // 2
    y1 = h // 2 - BOX_SIZE // 2
    x2 = x1 + BOX_SIZE
    y2 = y1 + BOX_SIZE

    # Extract the region of interest (hand area)
    roi = frame[y1:y2, x1:x2]

    # Run prediction
    letter, confidence = predict(roi)

    # Only type if confidence is high enough and cooldown has passed
    now = time.time()
    if confidence >= CONFIDENCE and (now - last_typed) >= COOLDOWN:
        if letter == "space":
            pyautogui.press("space")
            current_word += " "
        elif letter == "del":
            pyautogui.press("backspace")
            current_word = current_word[:-1]
        elif letter == "nothing":
            pass                                 # do nothing for "nothing" class
        else:
            pyautogui.typewrite(letter.lower())  # types into whatever is focused
            current_word += letter
        last_typed = now
        print(f"Typed: {letter} (confidence: {confidence:.2f})")

    # ── DRAW UI ───────────────────────────────────────────────────────────────
    # Draw the capture box
    box_color = (0, 255, 0) if confidence >= CONFIDENCE else (0, 165, 255)
    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

    # Show prediction and confidence
    cv2.putText(frame, f"Letter: {letter}",
                (x1, y1 - 40), cv2.FONT_HERSHEY_SIMPLEX, 1, box_color, 2)
    cv2.putText(frame, f"Confidence: {confidence:.2f}",
                (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, box_color, 2)

    # Show current word being built
    cv2.putText(frame, f"Word: {current_word}",
                (20, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    # Show cooldown bar
    elapsed  = min(now - last_typed, COOLDOWN)
    bar_w    = int((elapsed / COOLDOWN) * BOX_SIZE)
    cv2.rectangle(frame, (x1, y2 + 5), (x1 + bar_w, y2 + 15), (0, 255, 0), -1)
    cv2.putText(frame, "Ready" if bar_w == BOX_SIZE else "Wait...",
                (x1, y2 + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)

    cv2.imshow("ASL Sign Language Translator", frame)

    # Keyboard controls
    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break
    elif key == ord(" "):
        pyautogui.press("space")
        current_word += " "
    elif key == 8:                               # backspace key
        pyautogui.press("backspace")
        current_word = current_word[:-1]

cap.release()
cv2.destroyAllWindows()
print(f"\nSession ended. Full text typed: {current_word}")